# Lezione 5 — Transformers e il meccanismo di attenzione

## Obiettivi di apprendimento

- Capire i concetti di Query (Q), Key (K) e Value (V) nella self-attention.
- Vedere come si calcolano i punteggi di attenzione (prodotto scalare, scalatura, softmax).
- Implementare un layer di self-attention personalizzato in Keras.
- Costruire e addestrare un modello encoder-decoder (LSTM + attenzione) per la traduzione EN→IT.
- Capire perché questo modello NON è il Transformer canonico del paper originale.

## Cosa costruiremo

Un mini traduttore dall'inglese all'italiano addestrato con sole 5 frasi di esempio.
Non tradurrà nulla di utile nella vita reale (il dataset è troppo piccolo),
ma ci permette di vedere, con codice che viene eseguito davvero, come funziona
internamente l'attenzione.

## ⚠️ Avviso importante prima di iniziare

Il materiale originale di questo notebook si intitola "Transformers with Keras",
ma **non implementa il Transformer del paper "Attention Is All You Need"**
(quello usato realmente da GPT, Llama o BERT). Quello che costruiremo è un
modello ibrido più semplice:

- Un **encoder** e un **decoder** basati su **LSTM** (ricorrente), non su
  attenzione pura.
- Un unico layer di **self-attention personalizzato** che collega encoder e
  decoder — senza attenzione multi-testa, senza *positional encoding*.

È l'introduzione concettuale perfetta a Q/K/V, ma non confondere questa
architettura con quella di un vero LLM. Maggiori dettagli nel capitolo
[08_transformers_atencion.md](../manual/08_transformers_atencion_IT.md).

## Nota veloce: cosa sono Q, K e V?

- **Query (Q)**: la domanda che fa ogni parola: "chi mi aiuta a
  capire il mio contesto?".
- **Key (K)**: l'etichetta con cui ogni parola si presenta alle altre.
- **Value (V)**: l'informazione reale che porta quella parola.

Calcolo dell'attenzione: `scores = (Q · Kᵀ) / √dk`, poi `softmax(scores)`
per ottenere pesi che sommano a 1, e questi pesi ponderano i Values.

## Passo 0: installare le dipendenze

Esegui questa cella una sola volta (usa le stesse versioni del materiale sorgente).

In [ ]:
!pip install tensorflow_cpu==2.17.1
!pip install matplotlib==3.9.2

print("==== Librerie installate =====")

In [ ]:
import os
# Sopprime i messaggi di avviso di TensorFlow per l'uso della CPU
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [ ]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Concatenate, Layer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import backend as K
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter('ignore', FutureWarning)

## Passo 1: preparare i dati

Definiamo 5 coppie di frasi inglese-italiano. A ogni frase in italiano
aggiungiamo i token speciali `startseq` / `endseq`, che indicano al
decoder quando iniziare e quando finire di tradurre.

In [ ]:
# Coppie di frasi parallele (inglese -> italiano)
input_texts = [
    "Hello.", "How are you?", "I am learning machine translation.",
    "What is your name?", "I love programming."
]
target_texts = [
    "Ciao.", "Come stai?", "Sto imparando la traduzione automatica.",
    "Come ti chiami?", "Amo programmare."
]

# startseq/endseq dicono al decoder quando iniziare e quando fermarsi
target_texts = ["startseq " + x + " endseq" for x in target_texts]

In [ ]:
# Tokenizzazione: convertiamo le parole in numeri
input_tokenizer = Tokenizer()
input_tokenizer.fit_on_texts(input_texts)
input_sequences = input_tokenizer.texts_to_sequences(input_texts)

output_tokenizer = Tokenizer()
output_tokenizer.fit_on_texts(target_texts)
output_sequences = output_tokenizer.texts_to_sequences(target_texts)

input_vocab_size = len(input_tokenizer.word_index) + 1
output_vocab_size = len(output_tokenizer.word_index) + 1

print("Vocabolario inglese:", input_vocab_size, "parole")
print("Vocabolario italiano:", output_vocab_size, "parole")

In [ ]:
# Padding: tutte le sequenze devono avere la stessa lunghezza
max_input_length = max([len(seq) for seq in input_sequences])
max_output_length = max([len(seq) for seq in output_sequences])

input_sequences = pad_sequences(input_sequences, maxlen=max_input_length, padding='post')
output_sequences = pad_sequences(output_sequences, maxlen=max_output_length, padding='post')

# Il decoder riceve la frase italiana spostata di un passo (input / target)
decoder_input_data = output_sequences[:, :-1]
decoder_output_data = output_sequences[:, 1:]

# Convertiamo il target in one-hot (una probabilità per ogni parola del vocabolario)
decoder_output_data = np.array([np.eye(output_vocab_size)[seq] for seq in decoder_output_data])

## Passo 2: il layer di self-attention

In `build()` inizializziamo le matrici addestrabili `Wq`, `Wk`, `Wv`. In
`call()` calcoliamo Q, K, V, i punteggi di attenzione scalati, il
softmax, e l'output finale ponderato da V.

In [ ]:
class SelfAttention(Layer):
    def __init__(self, **kwargs):
        super(SelfAttention, self).__init__(**kwargs)

    def build(self, input_shape):
        # input_shape è una lista: [forma_di_q, forma_di_k, forma_di_v]
        feature_dim = input_shape[0][-1]

        self.Wq = self.add_weight(
            shape=(feature_dim, feature_dim),
            initializer='glorot_uniform',
            trainable=True,
            name='Wq'
        )
        self.Wk = self.add_weight(
            shape=(feature_dim, feature_dim),
            initializer='glorot_uniform',
            trainable=True,
            name='Wk'
        )
        self.Wv = self.add_weight(
            shape=(feature_dim, feature_dim),
            initializer='glorot_uniform',
            trainable=True,
            name='Wv'
        )
        super(SelfAttention, self).build(input_shape)

    def call(self, inputs):
        # inputs è una lista [query, key, value]
        q, k, v = inputs

        q = K.dot(q, self.Wq)
        k = K.dot(k, self.Wk)
        v = K.dot(v, self.Wv)

        # Attenzione scalata: (Q . K^T) / sqrt(dk)
        scores = K.batch_dot(q, k, axes=[2, 2])
        dk = K.cast(K.shape(k)[-1], dtype=K.floatx())
        scores = scores / K.sqrt(dk)

        # softmax normalizza i punteggi in pesi che sommano a 1
        attention_weights = K.softmax(scores, axis=-1)
        output = K.batch_dot(attention_weights, v)

        return output

## Passo 3: architettura encoder-decoder + attenzione

- **Encoder**: Embedding + LSTM sulla frase in inglese. Genera un
  "riassunto" (stati `state_h`, `state_c`).
- **Decoder**: Embedding + LSTM sulla frase in italiano, inizializzato con
  il riassunto dell'encoder.
- **Attenzione**: il decoder "consulta" gli output dell'encoder a ogni passo.
- **Dense finale**: predice la parola successiva in italiano (softmax su
  tutto il vocabolario).

In [ ]:
# --- Encoder ---
encoder_inputs = Input(shape=(max_input_length,))
encoder_embedding = Embedding(input_vocab_size, 256)(encoder_inputs)
encoder_lstm = LSTM(256, return_sequences=True, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)
encoder_states = [state_h, state_c]

# --- Decoder ---
decoder_inputs = Input(shape=(max_output_length - 1,))
decoder_embedding = Embedding(output_vocab_size, 256)(decoder_inputs)
decoder_lstm = LSTM(256, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

# --- Attenzione: il decoder consulta gli output dell'encoder ---
self_attention = SelfAttention()
attention_output = self_attention([decoder_outputs, encoder_outputs, encoder_outputs])

# Combiniamo l'output del decoder con il contesto di attenzione
decoder_concat = Concatenate(axis=-1)([decoder_outputs, attention_output])

# Layer finale: predice la parola in italiano
decoder_dense = Dense(output_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_concat)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

## Passo 4: addestrare il modello

Con un dataset di 5 frasi, 100 epoche si addestrano in pochi secondi sulla CPU
gratuita di Colab — un numero di epoche perfettamente ragionevole qui perché
il dataset è minuscolo (non farlo con dataset grandi senza GPU).

In [ ]:
history_glorot_adam = model.fit(
    [input_sequences, decoder_input_data],
    decoder_output_data,
    epochs=100,
    batch_size=16
)

## Passo 5: rappresentare graficamente la perdita di addestramento

In [ ]:
plt.plot(history_glorot_adam.history['loss'])
plt.title('Perdita di addestramento')
plt.xlabel('Epoche')
plt.ylabel('Perdita')
plt.show()

## Riepilogo della lezione

- La self-attention si calcola con tre matrici addestrabili `Wq`, `Wk`, `Wv`.
- I punteggi di attenzione sono `(Q · Kᵀ) / √dk`, normalizzati con softmax.
- Questo modello combina LSTM (encoder-decoder) con un layer di attenzione — **non**
  è il Transformer canonico del paper originale.
- La perdita (loss) dovrebbe scendere in modo consistente lungo le 100
  epoche, segno che il modello sta imparando ad associare le frasi.

## Per continuare

- Ripassa la teoria completa in
  [manual/08_transformers_atencion.md](../manual/08_transformers_atencion_IT.md).
- Metti in pratica quanto imparato in
  [clase05_practica_transformers_atencion.ipynb](../notebooks_practica/clase05_practica_transformers_atencion_IT.ipynb),
  dove confronterai l'effetto del cambio dell'inizializzatore dei pesi.